# 01 · T4 benchmark: memory and speed for LoRA on AfriMed-QA

**Purpose (PLAN.md, step 2):** measure GPU memory and training/inference speed for the candidate models before choosing a model size. This notebook does **not** train a real model or produce the baseline.

**How to run**
1. *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*. When asked, upload `qa_train.jsonl` and `qa_val.jsonl` from `data/processed/` (made by `scripts/prepare_data.py`).
3. Send back the JSON printed by the last cell. It is also saved as `t4_benchmark_results.json`.

**What it measures**
- Token lengths of the formatted examples, to choose `max_len`.
- For each model and each (max_len, batch size) setting: peak GPU memory, seconds per step, throughput, and estimated time per epoch. Out-of-memory runs are recorded, not fatal.
- GPU inference speed: one forward pass for each multiple-choice question, and generated short answers.
- CPU inference speed. The Colab CPU is a rough stand-in for the free HF Spaces CPU.

It takes about 20–30 minutes on a T4.

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q peft

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
import gc, hashlib, json, math, os, platform, random, time
import numpy as np
import torch
import transformers, peft
from packaging import version
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > T4 GPU"
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "gpu_mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "python": platform.python_version(),
    "cpu_count": os.cpu_count(),
}
print(json.dumps(ENV, indent=2))

{
  "gpu": "Tesla T4",
  "gpu_mem_gb": 15.6,
  "torch": "2.11.0+cu130",
  "transformers": "5.18.0",
  "peft": "0.21.1",
  "python": "3.13.15",
  "cpu_count": 2
}


## Config

In [ ]:
MODELS = ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"]
GRID = [(512, 4), (512, 8), (1024, 2), (1024, 4)]   # (max_len, micro-batch size)
WARMUP_STEPS, MEASURE_STEPS = 3, 20
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
LR = 2e-4
N_MCQ_INFER, N_SAQ_INFER, SAQ_MAX_NEW = 50, 20, 128
RUN_CPU = True            # CPU inference timing (slow but useful for the HF Spaces decision)
N_CPU_PROMPTS, CPU_MAX_NEW = 3, 64
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Hashes of the files written by scripts/prepare_data.py (seed 42), so we know it is the same split.
EXPECTED_SHA256 = {
    "qa_train.jsonl": "17c8393f77d4d5773869a9063638532e843a08a932017725618ac342962f828e",
    "qa_val.jsonl": "fdefd6ac7619b53b689096bd5d0ffa63ae62596223450cece4ed7992a07aa0a9",
}
# T4 has no bf16 support, so fp16 autocast is used.
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"

## Data

In [ ]:
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
missing = [f for f in EXPECTED_SHA256 if not os.path.exists(f"{DATA_DIR}/{f}")]
if missing:
    from google.colab import files
    print("Upload:", missing)
    for name, content in files.upload().items():
        with open(f"{DATA_DIR}/{os.path.basename(name)}", "wb") as fh:
            fh.write(content)

for f, expected in EXPECTED_SHA256.items():
    digest = hashlib.sha256(open(f"{DATA_DIR}/{f}", "rb").read()).hexdigest()
    status = "OK" if digest == expected else "MISMATCH (different split than prepare_data.py seed 42)"
    print(f"{f}: {status}")

load = lambda f: [json.loads(l) for l in open(f"{DATA_DIR}/{f}", encoding="utf-8")]
train, val = load("qa_train.jsonl"), load("qa_val.jsonl")
print(len(train), "train /", len(val), "val")

Upload: ['qa_train.jsonl', 'qa_val.jsonl']


Saving qa_train.jsonl to qa_train.jsonl
Saving qa_val.jsonl to qa_val.jsonl
qa_train.jsonl: OK
qa_val.jsonl: OK
3685 train / 526 val


## Prompt format

A first version, to be moved to `src/` for the real training in step 4.
- **MCQ:** the options are shown as A–E. The target is `Answer: B`, or `Answer: B, D` when several options are correct.
- **SAQ:** the target is the reference answer text.
- **Loss** is computed on the assistant tokens only.

In [ ]:
SYSTEM = ("You are a medical question-answering assistant focused on healthcare in African contexts. "
          "Answer accurately and concisely. You do not replace a qualified health professional.")
LETTERS = "ABCDE"

def mcq_user(rec):
    opts = "\n".join(f"{LETTERS[i]}. {rec['options'][f'option{i + 1}']}" for i in range(len(rec["options"])))
    return (f"{rec['question']}\n\nOptions:\n{opts}\n\n"
            "Select the correct option(s). Reply in the form 'Answer: <letter(s)>'.")

def mcq_target(rec):
    return "Answer: " + ", ".join(LETTERS[int(k.replace("option", "")) - 1] for k in rec["answer"])

def to_messages(rec, with_answer=True):
    if rec["question_type"] == "mcq":
        user, target = mcq_user(rec), mcq_target(rec)
    else:
        user, target = rec["question"], rec["answer"]
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return msgs + [{"role": "assistant", "content": target}] if with_answer else msgs

def encode(tok, rec, max_len=None):
    # Render to text first: apply_chat_template's tokenize=True return type differs across versions.
    prompt = tok.apply_chat_template(to_messages(rec, False), tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(to_messages(rec), tokenize=False)
    assert full.startswith(prompt), "chat template is not prefix-stable"
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    truncated = max_len is not None and len(f_ids) > max_len
    if truncated:
        f_ids, labels = f_ids[:max_len], labels[:max_len]
    return {"input_ids": f_ids, "labels": labels, "truncated": truncated}

tok = AutoTokenizer.from_pretrained(MODELS[0])
print(tok.apply_chat_template(to_messages(next(r for r in train if r["question_type"] == "mcq")), tokenize=False))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are a medical question-answering assistant focused on healthcare in African contexts. Answer accurately and concisely. You do not replace a qualified health professional.<|im_end|>
<|im_start|>user
Which of the following conditions is characterized by an abnormal enlargement of the thyroid gland, often resulting in a visibly enlarged neck, and is endemic in certain regions of Africa due to dietary iodine deficiency?

Options:
A. Hashimoto's thyroiditis
B. Graves' disease
C. Toxic multinodular goiter
D. Simple (nontoxic) goiter
E. Thyroid cancer

Select the correct option(s). Reply in the form 'Answer: <letter(s)>'.<|im_end|>
<|im_start|>assistant
Answer: D<|im_end|>



## Token lengths (Qwen2.5 sizes share one tokenizer)

In [ ]:
TOKEN_STATS = {}
for qtype in ["mcq", "saq", "all"]:
    lens = np.array([len(encode(tok, r)["input_ids"]) for r in train if qtype == "all" or r["question_type"] == qtype])
    TOKEN_STATS[qtype] = {
        "n": int(len(lens)),
        **{f"p{p}": int(np.percentile(lens, p)) for p in (50, 90, 95, 99)},
        "max": int(lens.max()),
        **{f"pct_over_{L}": round(float((lens > L).mean() * 100), 2) for L in (256, 512, 1024)},
    }
print(json.dumps(TOKEN_STATS, indent=2))

{
  "mcq": {
    "n": 2824,
    "p50": 139,
    "p90": 193,
    "p95": 211,
    "p99": 257,
    "max": 462,
    "pct_over_256": 1.06,
    "pct_over_512": 0.0,
    "pct_over_1024": 0.0
  },
  "saq": {
    "n": 861,
    "p50": 108,
    "p90": 194,
    "p95": 292,
    "p99": 440,
    "max": 1213,
    "pct_over_256": 6.39,
    "pct_over_512": 0.58,
    "pct_over_1024": 0.12
  },
  "all": {
    "n": 3685,
    "p50": 134,
    "p90": 193,
    "p95": 218,
    "p99": 338,
    "max": 1213,
    "pct_over_256": 2.31,
    "pct_over_512": 0.14,
    "pct_over_1024": 0.03
  }
}


## Training benchmark

In [ ]:
def free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def load_lora_model(name):
    model = AutoModelForCausalLM.from_pretrained(name, **{DTYPE_KW: torch.float16}).cuda()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(**LORA))
    for p in model.parameters():          # fp32 adapters for stable fp16 mixed precision
        if p.requires_grad:
            p.data = p.data.float()
    return model

def collate(examples, pad_id):
    L = max(len(e["input_ids"]) for e in examples)
    ids = torch.full((len(examples), L), pad_id)
    labels = torch.full((len(examples), L), -100)
    mask = torch.zeros((len(examples), L), dtype=torch.long)
    for i, e in enumerate(examples):
        n = len(e["input_ids"])
        ids[i, :n] = torch.tensor(e["input_ids"]); labels[i, :n] = torch.tensor(e["labels"]); mask[i, :n] = 1
    return ids.cuda(), labels.cuda(), mask.cuda()

def bench_train(name, max_len, bs):
    free()
    model = load_lora_model(name)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    scaler = torch.amp.GradScaler("cuda")
    encoded = [encode(tok, r, max_len) for r in train]
    rng = random.Random(SEED)          # same random batches for every config
    losses, tokens, t0 = [], 0, None
    try:
        for step in range(WARMUP_STEPS + MEASURE_STEPS):
            if step == WARMUP_STEPS:
                torch.cuda.synchronize(); t0 = time.perf_counter(); tokens = 0
            ids, labels, mask = collate(rng.sample(encoded, bs), tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            losses.append(loss.item()); tokens += int(mask.sum())
        torch.cuda.synchronize()
        sec_step = (time.perf_counter() - t0) / MEASURE_STEPS
        res = {
            "status": "ok",
            "sec_per_step": round(sec_step, 3),
            "examples_per_sec": round(bs / sec_step, 2),
            "tokens_per_sec": round(tokens / MEASURE_STEPS / sec_step),
            "peak_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "est_min_per_epoch": round(len(train) / (bs / sec_step) / 60, 1),
            "loss_first_last": [round(losses[0], 3), round(losses[-1], 3)],
            "trainable_params_m": round(n_trainable / 1e6, 2),
            "pct_truncated": round(100 * sum(e["truncated"] for e in encoded) / len(encoded), 2),
        }
    except torch.cuda.OutOfMemoryError:
        res = {"status": "OOM"}
    del model, opt
    free()
    return {"model": name, "max_len": max_len, "batch_size": bs, **res}

TRAIN_RESULTS = []
for name in MODELS:
    for max_len, bs in GRID:
        r = bench_train(name, max_len, bs)
        TRAIN_RESULTS.append(r)
        print(r)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'max_len': 512, 'batch_size': 4, 'status': 'ok', 'sec_per_step': 0.223, 'examples_per_sec': 17.93, 'tokens_per_sec': 2423, 'peak_mem_gb': 4.95, 'est_min_per_epoch': 3.4, 'loss_first_last': [1.562, 1.347], 'trainable_params_m': 8.8, 'pct_truncated': 0.14}


...

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'max_len': 512, 'batch_size': 8, 'status': 'OOM'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'max_len': 1024, 'batch_size': 2, 'status': 'ok', 'sec_per_step': 0.263, 'examples_per_sec': 7.61, 'tokens_per_sec': 1011, 'peak_mem_gb': 6.3, 'est_min_per_epoch': 8.1, 'loss_first_last': [1.042, 0.227], 'trainable_params_m': 18.46, 'pct_truncated': 0.03}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'max_len': 1024, 'batch_size': 4, 'status': 'ok', 'sec_per_step': 0.424, 'examples_per_sec': 9.44, 'tokens_per_sec': 1275, 'peak_mem_gb': 9.19, 'est_min_per_epoch': 6.5, 'loss_first_last': [1.352, 1.107], 'trainable_params_m': 18.46, 'pct_truncated': 0.03}


In [ ]:
import pandas as pd
pd.DataFrame(TRAIN_RESULTS)

,model,max_len,batch_size,status,sec_per_step,examples_per_sec,tokens_per_sec,peak_mem_gb,est_min_per_epoch,loss_first_last,trainable_params_m,pct_truncated
0,Qwen/Qwen2.5-0.5B-Instruct,512,4,ok,0.223,17.93,2423.0,4.95,3.4,"[1.562, 1.347]",8.80,0.14
1,Qwen/Qwen2.5-0.5B-Instruct,512,8,ok,0.400,20.02,2772.0,9.65,3.1,"[1.643, 2.237]",8.80,0.14
2,Qwen/Qwen2.5-0.5B-Instruct,1024,2,ok,0.208,9.59,1276.0,3.04,6.4,"[1.405, 0.261]",8.80,0.03
3,Qwen/Qwen2.5-0.5B-Instruct,1024,4,ok,0.212,18.90,2553.0,4.94,3.3,"[1.562, 1.348]",8.80,0.03
4,Qwen/Qwen2.5-1.5B-Instruct,512,4,ok,0.409,9.77,1320.0,9.19,6.3,"[1.352, 1.103]",18.46,0.14
5,Qwen/Qwen2.5-1.5B-Instruct,512,8,OOM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Qwen/Qwen2.5-1.5B-Instruct,1024,2,ok,0.263,7.61,1011.0,6.30,8.1,"[1.042, 0.227]",18.46,0.03
7,Qwen/Qwen2.5-1.5B-Instruct,1024,4,ok,0.424,9.44,1275.0,9.19,6.5,"[1.352, 1.107]",18.46,0.03


## GPU inference benchmark (base model, no adapter)

A merged LoRA model runs at the same speed as the base model. The 50-question MCQ accuracy below is only a sanity check, **not** the baseline result.

In [ ]:
def mcq_letter_ids():
    # Token ids for " A".."E" and "A".."E"; whichever the model prefers after "Answer:".
    return [[tok(f" {c}", add_special_tokens=False)["input_ids"][0], tok(c, add_special_tokens=False)["input_ids"][0]]
            for c in LETTERS]

@torch.no_grad()
def bench_infer_gpu(name):
    free()
    model = AutoModelForCausalLM.from_pretrained(name, **{DTYPE_KW: torch.float16}).cuda().eval()
    tok.padding_side = "left"
    letter_ids = mcq_letter_ids()

    mcq = [r for r in val if r["question_type"] == "mcq" and len(r["answer"]) == 1][:N_MCQ_INFER]
    correct = 0
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for r in mcq:
        prompt = tok.apply_chat_template(to_messages(r, False), tokenize=False, add_generation_prompt=True) + "Answer:"
        ids = tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.cuda()
        logits = model(ids).logits[0, -1].float()
        scores = [max(logits[i].item() for i in pair) for pair in letter_ids]
        correct += LETTERS[int(np.argmax(scores))] == mcq_target(r)[-1]
    torch.cuda.synchronize(); mcq_sec = (time.perf_counter() - t0) / len(mcq)

    saq = [r for r in val if r["question_type"] == "saq"][:N_SAQ_INFER]
    gen_tokens, t0 = 0, time.perf_counter()
    samples = []
    for r in saq:
        prompt = tok.apply_chat_template(to_messages(r, False), tokenize=False, add_generation_prompt=True)
        enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=SAQ_MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        new = out[0, enc.input_ids.shape[1]:]
        gen_tokens += len(new)
        if len(samples) < 3:
            samples.append({"q": r["question"][:200], "ref": r["answer"][:200],
                            "zero_shot": tok.decode(new, skip_special_tokens=True)[:300]})
    torch.cuda.synchronize(); saq_total = time.perf_counter() - t0
    del model; free()
    tok.padding_side = "right"
    return {"model": name, "device": "cuda",
            "mcq_sec_per_q": round(mcq_sec, 3), "mcq_sanity_acc": f"{correct}/{len(mcq)}",
            "saq_sec_per_answer": round(saq_total / len(saq), 2),
            "saq_gen_tokens_per_sec": round(gen_tokens / saq_total, 1),
            "saq_avg_new_tokens": round(gen_tokens / len(saq), 1),
            "zero_shot_samples": samples}

INFER_RESULTS = [bench_infer_gpu(n) for n in MODELS]
for r in INFER_RESULTS:
    print(json.dumps(r, indent=2))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "device": "cuda",
  "mcq_sec_per_q": 0.043,
  "mcq_sanity_acc": "22/50",
  "saq_sec_per_answer": 4.09,
  "saq_gen_tokens_per_sec": 27.7,
  "saq_avg_new_tokens": 113.2,
  "zero_shot_samples": [
    {
      "q": "How does the management of tuberculosis differ in HIV-positive individuals in Africa compared to those in developed countries?",
      "ref": "In Africa, where HIV and tuberculosis co-infection rates are high, there is an increased emphasis on screening for and treating both conditions simultaneously.",
      "zero_shot": "The management of tuberculosis (TB) differs significantly between HIV-positive individuals in Africa and those in developed countries due to several factors:\n\n1. **Prevalence**: TB is more prevalent among people living with HIV, particularly in sub-Saharan Africa where HIV prevalence rates are highe"
    },
    {
      "q": "What are the unique cultural beliefs and practices surrounding mental health in various Af

## CPU inference (a rough stand-in for HF Spaces free CPU)

Colab's free CPU has about 2 vCPUs, close to the free HF Spaces CPU tier. Treat these numbers as an order of magnitude only.

In [ ]:
@torch.no_grad()
def bench_infer_cpu(name):
    gc.collect()
    model = AutoModelForCausalLM.from_pretrained(name, **{DTYPE_KW: torch.float32}).eval()
    saq = [r for r in val if r["question_type"] == "saq"][:N_CPU_PROMPTS]
    gen_tokens, t0 = 0, time.perf_counter()
    for r in saq:
        prompt = tok.apply_chat_template(to_messages(r, False), tokenize=False, add_generation_prompt=True)
        enc = tok(prompt, return_tensors="pt", add_special_tokens=False)
        out = model.generate(**enc, max_new_tokens=CPU_MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        gen_tokens += out.shape[1] - enc.input_ids.shape[1]
    total = time.perf_counter() - t0
    del model; gc.collect()
    return {"model": name, "device": "cpu", "threads": torch.get_num_threads(),
            "sec_per_answer": round(total / len(saq), 1),
            "gen_tokens_per_sec": round(gen_tokens / total, 2),
            "max_new_tokens": CPU_MAX_NEW}

CPU_RESULTS = [bench_infer_cpu(n) for n in MODELS] if RUN_CPU else []
for r in CPU_RESULTS:
    print(r)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Results: send this JSON back

In [ ]:
RESULTS = {"env": ENV, "config": {"grid": GRID, "lora": LORA, "lr": LR,
                                  "warmup_steps": WARMUP_STEPS, "measure_steps": MEASURE_STEPS},
           "token_stats": TOKEN_STATS, "train": TRAIN_RESULTS,
           "infer_gpu": INFER_RESULTS, "infer_cpu": CPU_RESULTS}
with open("t4_benchmark_results.json", "w") as f:
    json.dump(RESULTS, f, indent=2)
print(json.dumps(RESULTS, indent=2))
try:
    from google.colab import files
    files.download("t4_benchmark_results.json")
except Exception as e:
    print("Download skipped:", e)